# 03 — Location forecast POC: weather × địa điểm × ngành × ngày

## tl;dr

POC gợi ý ZIP để khảo sát mở cửa hàng bằng **weather, địa điểm, ngành và ngày**.
So sánh ba model: calendar mean baseline, Poisson regression, Histogram Gradient
Boosting. Không dùng activity, lag, trend hay số business làm input dự đoán.
Activity chỉ dùng để chọn cohort trong training, tạo nhãn và chấm sai số.
Output là **check-in ghi nhận / cohort business / ngày**, không phải doanh thu
hay xác suất thành công của cửa hàng mới.

Chạy **Restart Kernel and Run All**; explore bằng `recommend(date, metro,
category, weather=...)`. Có thể nhập bất kỳ ngày nào nếu cung cấp weather cho
ngày đó. Bỏ `weather` để lấy observed weather từ snapshot, nếu ngày/station đủ dữ
liệu. Source không giữ output; bản đã chạy và CSV nằm trong
`data/processed/location_forecast_poc/`. Không cần database hoặc download.


Kết quả **cấu hình mặc định đã chạy**: 517 area/category groups, 20,793
business/category rows; 1,321,452 daily labels, 1,295,519 ngày đủ weather để
train/chấm. Không loại năm 2013 để chờ activity lag.

| Model | Validation 2018 deviance | Test 2019 deviance | Test MAE (check-in/business/ngày) |
| --- | ---: | ---: | ---: |
| calendar_mean | 0.05982 | 0.05664 | 0.04110 |
| hgb_weather | 0.06056 | 0.05880 | 0.04297 |
| poisson_weather | 0.09346 | 0.08884 | 0.05935 |

Baseline thắng overall; HGB được chọn trong hai model weather cho default
scenario. Weather model chưa vượt baseline, nên chưa có bằng chứng weather
cải thiện accuracy. Không có activity 2026 để kiểm tra sai số năm 2026.

Các sample đã chạy bằng HGB (rate là recorded check-in/cohort business/ngày):

| Ngày | Metro / ngành | Candidate đủ weather | Top ZIP | Predicted rate |
| --- | --- | ---: | --- | ---: |
| 2026-07-15 | Philadelphia / Restaurants | 299/299 | 19107 | 0.1641 |
| 2026-01-15 | Nashville / Restaurants | 44/44 | 37201 | 0.1403 |
| 2026-09-15 | Tampa / Day Spas | 11/11 | 33629 | 0.0642 |
| 2026-12-15 | Tampa / Day Spas, supplied scenario | 11/11 | 33629 | 0.0654 |

Prediction API đã kiểm tra ngày tương lai 2027 và leap day 2028 với weather
nhập; xóa/sửa activity truth không đổi prediction. Đổi config thì đọc kết quả
mới trong Results, không dùng các số mặc định này như kết quả scope khác.


## Context & Methods

- Grain: **metro group × source city/state/ZIP × ngành × ngày**. Ranking theo
  predicted daily check-in rate, giúp tránh xếp ZIP lớn lên đầu chỉ vì nhiều shop.
- Cohort chọn từ **2013–2016**; train **2013–2017**, validation **2018**, refit đến
  hết **2018**, test **2019**. Không random split, không grid search. 2019 từng
  được xem trong nghiên cứu trước; đây là holdout khám phá, chưa confirmatory.
- `X` chỉ chứa weather, tọa độ khu vực, metro, ngành và calendar từ ngày cần
  predict. `y` là activity thực tế. `n_businesses` chỉ dùng chuẩn hóa nhãn và
  training/scoring weights; không truyền vào model hoặc hàm predict.
- Validation/test dùng **observed target-day weather**, cùng hợp đồng input với
  inference. Chỉ chấm các ngày có đủ ba weather variables cho cả ba model.
  Đây là conditional prediction với weather đã biết, không phải đánh giá khả
  năng tự dự báo thời tiết. Ngày tương lai dùng weather forecast/scenario nhập vào.

### Key Assumptions

- Metro là county group có dữ liệu Yelp, không phải ranh giới CBSA chính thức.
  Candidate giữ source city/state/ZIP theo [area grain](../docs/adr/0001-compare-source-zip-areas.md).
  Tọa độ khu vực = mean tọa độ snapshot businesses thuộc training cohort.
- Không dùng `stars`, `review_count`, `is_open`, ngày check-in cuối hoặc nhãn
  validation/test để chọn cohort. Cohort count không phải số shop đang mở năm 2026.
- Yelp timestamps không có timezone được xác minh; ghép **source date label**
  với ngày station observation. Hai bên chưa chắc cùng cửa sổ 24 giờ, nên không
  kết luận tác động nhân quả của weather; daily alignment là hạn chế của POC.
- Dùng dominant station của mỗi area/ngành. Chỉ nhận weather numeric finite,
  flag blank/`T`; không tự fill missing observed day bằng normal.
- Zero = không có check-in ghi nhận, không phải shop đóng cửa. Chưa có nhãn
  thành công cửa hàng mới, mặt bằng, rent, footfall hay inventory hiện tại.
  Chỉ rank khu vực có training cohort; không tổng quát sang ZIP chưa được học.


### 1. Setup

Từ root: `uv sync --frozen --group notebooks`, rồi
`uv run --frozen --group notebooks jupyter lab --notebook-dir=.`.
Dùng kernel `.venv/bin/python`. Chạy notebook 01/02 trước nếu thiếu raw files.


In [ ]:
import hashlib
import importlib.metadata
import json
import re
from collections import Counter
from datetime import UTC, datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import PoissonRegressor
from sklearn.metrics import mean_absolute_error, mean_poisson_deviance
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from threadpoolctl import threadpool_limits

from sitesense.import_data.yelp import categories_from_source

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
pd.set_option("display.max_columns", 14)
pd.set_option("display.max_rows", 15)
plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True})
thread_limits = threadpool_limits(limits=2)
versions = {
    name: importlib.metadata.version(name)
    for name in ["sitesense", "numpy", "pandas", "scikit-learn", "nbformat"]
}
display(pd.Series(versions, name="installed version"))

### 2. Cấu hình

POC mặc định Restaurants và Day Spas trong ba metro. County FIPS/ZIP là chuỗi.
Đổi scope hoặc date split thì chạy lại từ đầu và đọc bảng coverage mới.


In [ ]:
METRO_COUNTIES = {
    "Philadelphia": [
        "42017",
        "42029",
        "42045",
        "42091",
        "42101",
        "34005",
        "34007",
        "34015",
        "34033",
        "10003",
    ],
    "Nashville": ["47021", "47037", "47147", "47149", "47165", "47187", "47189"],
    "Tampa": ["12053", "12057", "12101", "12103"],
}
CATEGORIES = ["Restaurants", "Day Spas"]
START = pd.Timestamp("2013-01-01")
COHORT_END = pd.Timestamp("2016-12-31")
VALIDATION_START = pd.Timestamp("2018-01-01")
TEST_START = pd.Timestamp("2019-01-01")
END = pd.Timestamp("2019-12-31")
MIN_BUSINESSES = 5
MIN_TRAIN_CHECKINS = 100
TOP_K = 3
SEED = 42
WEATHER = ["tmax_c", "tmin_c", "precip_mm"]
SOURCE_NOTEBOOK = ROOT / "notebooks/03_location_forecast_poc.ipynb"
RUN_DIR = (
    ROOT
    / "data/processed/location_forecast_poc"
    / datetime.now(UTC).strftime("run_%Y%m%dT%H%M%S%fZ")
)
assert START <= COHORT_END < VALIDATION_START < TEST_START <= END
assert MIN_BUSINESSES >= 1
assert len(CATEGORIES) == len(set(CATEGORIES)) and TOP_K >= 1
county_to_metro = {fips: metro for metro, counties in METRO_COUNTIES.items() for fips in counties}
assert len(county_to_metro) == sum(map(len, METRO_COUNTIES.values()))

## Data

### 3. Kiểm tra provenance

So SHA-256 với manifest của notebook 01/02. Dùng daily ACIS station observations
trong Climate archive, không dùng county projection để thay weather.


In [ ]:
paths = {
    "business": ROOT / "data/raw/yelp/business.jsonl",
    "checkin": ROOT / "data/raw/yelp/checkin.jsonl",
    "yelp_manifest": ROOT / "data/raw/yelp/staging_manifest.json",
    "mapping": ROOT / "data/raw/climate/yelp_scope/business_climate_mapping.csv",
    "scope_manifest": ROOT / "data/raw/climate/yelp_scope/scope_manifest.json",
    "observations": ROOT / "data/raw/climate/yelp_stations/observations.csv.gz",
    "station_manifest": ROOT / "data/raw/climate/yelp_stations/manifest.json",
}
for path in paths.values():
    if not path.is_file():
        raise FileNotFoundError(f"Missing {path.relative_to(ROOT)}. Run notebooks 01 and 02.")


def sha256(path):
    with path.open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()


source_hashes = {name: sha256(path) for name, path in paths.items()}
yelp_manifest = json.loads(paths["yelp_manifest"].read_text())
station_manifest = json.loads(paths["station_manifest"].read_text())
scope_manifest = json.loads(paths["scope_manifest"].read_text())
assert station_manifest["status"] == "complete"
for name in ["business", "checkin"]:
    assert source_hashes[name] == yelp_manifest["staged_sha256"][name]
    assert paths[name].stat().st_size == yelp_manifest["staged_bytes"][name]
assert source_hashes["observations"] == station_manifest["csv"]["sha256"]
expected_mapping = next(
    item["sha256"]
    for item in scope_manifest["outputs"]
    if item["path"].endswith("/business_climate_mapping.csv")
)
assert source_hashes["mapping"] == expected_mapping
source_audit = pd.DataFrame(
    [
        {"source": name, "path": str(path.relative_to(ROOT)), "sha256": source_hashes[name]}
        for name, path in paths.items()
    ]
)
display(source_audit)

### 4. Địa điểm và ngành

Giữ source city/state/ZIP, normalize whitespace/case. Tọa độ là metadata địa điểm,
không có activity. Các dòng thiếu ZIP hoặc tọa độ hợp lệ được report.


In [ ]:
mapping = pd.read_csv(paths["mapping"], dtype=str, keep_default_na=False)
assert mapping["business_id"].is_unique and len(mapping) == scope_manifest["businesses"]
mapping["metro"] = mapping["county_fips"].map(county_to_metro)
metro_mapping = mapping[mapping["metro"].notna() & mapping["status"].eq("mapped")]
lookup = metro_mapping.set_index("business_id")
records = []
with paths["business"].open() as handle:
    for line in handle:
        item = json.loads(line)
        business_id = item["business_id"]
        if business_id not in lookup.index:
            continue
        labels = categories_from_source(item.get("categories"))
        for category in CATEGORIES:
            if category in labels:
                place = lookup.loc[business_id]
                city = " ".join(item["city"].split()).casefold()
                state = item["state"].strip().upper()
                zip_code = item["postal_code"].strip()
                records.append(
                    {
                        "business_id": business_id,
                        "metro": place["metro"],
                        "city": city,
                        "state": state,
                        "zip": zip_code,
                        "category": category,
                        "latitude": float(item["latitude"]),
                        "longitude": float(item["longitude"]),
                        "station_id": place["station_id"],
                        "station_distance_km": float(place["station_distance_km"]),
                    }
                )
businesses = pd.DataFrame(records)
assert not businesses.empty, "No matching category/metro businesses."
assert not businesses.duplicated(["business_id", "category"]).any()
excluded_invalid_zip = int((~businesses["zip"].str.fullmatch(r"[0-9]{5}")).sum())
valid_coordinates = (
    np.isfinite(businesses[["latitude", "longitude"]]).all(axis=1)
    & businesses["latitude"].between(-90, 90)
    & businesses["longitude"].between(-180, 180)
)
excluded_invalid_coordinates = int((~valid_coordinates).sum())
businesses = businesses[businesses["zip"].str.fullmatch(r"[0-9]{5}") & valid_coordinates].copy()
businesses["area_id"] = businesses[["metro", "state", "city", "zip"]].agg("|".join, axis=1)
businesses["group_id"] = businesses["area_id"] + "|" + businesses["category"]
display(
    businesses.groupby(["metro", "category"]).agg(
        snapshot_businesses=("business_id", "size"), source_areas=("area_id", "nunique")
    )
)
print(f"Excluded business/category rows with invalid ZIP: {excluded_invalid_zip:,}")
print(f"Excluded invalid coordinate rows: {excluded_invalid_coordinates:,}")

### 5. Chọn cohort chỉ bằng training activity

Pass đầu stream check-in để chọn business có activity 2013–2016. Minimum 5
businesses và 100 training check-ins mỗi area/ngành. Dominant station và tọa độ
được cố định từ cohort này. Activity validation/test không tham gia selection.


In [ ]:
selected_ids = set(businesses["business_id"])
training_counts = Counter()
seen_checkin_ids = set()
start_label, cohort_end_label = START.strftime("%Y-%m-%d"), COHORT_END.strftime("%Y-%m-%d")
with paths["checkin"].open() as handle:
    for line in handle:
        item = json.loads(line)
        business_id = item["business_id"]
        if business_id not in selected_ids:
            continue
        assert business_id not in seen_checkin_ids
        seen_checkin_ids.add(business_id)
        for timestamp in item["date"].split(", "):
            if start_label <= timestamp[:10] <= cohort_end_label:
                assert re.fullmatch(r"[0-9]{4}-[0-9]{2}-[0-9]{2} [0-9:]{8}", timestamp)
                training_counts[business_id] += 1
training_activity = pd.Series(training_counts, name="train_checkins")
cohort = businesses[businesses["business_id"].isin(training_activity.index)].copy()
cohort["train_checkins"] = cohort["business_id"].map(training_activity)
area_stats = cohort.groupby("group_id").agg(
    n_businesses=("business_id", "size"), train_checkins=("train_checkins", "sum")
)
eligible = area_stats[
    (area_stats["n_businesses"] >= MIN_BUSINESSES)
    & (area_stats["train_checkins"] >= MIN_TRAIN_CHECKINS)
]
cohort = cohort[cohort["group_id"].isin(eligible.index)].copy()
assert not cohort.empty, "Lower training coverage thresholds or widen scope."
station_counts = cohort.groupby(["group_id", "station_id"]).size().rename("station_businesses")
station_choice = (
    station_counts.reset_index()
    .sort_values(["group_id", "station_businesses", "station_id"], ascending=[True, False, True])
    .drop_duplicates("group_id")
    .set_index("group_id")
)
areas = cohort.groupby("group_id")[["area_id", "metro", "city", "state", "zip", "category"]].first()
coordinates = cohort.groupby("group_id")[["latitude", "longitude"]].mean()
areas = areas.join(coordinates).join(eligible).join(station_choice)
areas["station_share"] = areas["station_businesses"] / areas["n_businesses"]
cohort_summary = cohort.groupby(["metro", "category"]).agg(
    areas=("group_id", "nunique"),
    cohort_businesses=("business_id", "size"),
    train_checkins=("train_checkins", "sum"),
    station_distance_p95_km=("station_distance_km", lambda x: x.quantile(0.95)),
)
display(cohort_summary)
print(f"{len(areas):,} area/category groups; {len(cohort):,} cohort business/category rows")

### 6. Daily labels và calendar

Pass thứ hai aggregate thẳng về area/ngành/ngày, tránh materialize business×day.
Cùng cohort cho mọi ngày, fill zero activity. Không tạo lag/rolling/trend.


In [ ]:
business_groups = cohort.groupby("business_id")["group_id"].agg(list).to_dict()
daily_counts = Counter()
end_label = END.strftime("%Y-%m-%d")
recorded_events = 0
with paths["checkin"].open() as handle:
    for line in handle:
        item = json.loads(line)
        groups = business_groups.get(item["business_id"])
        if groups is None:
            continue
        for timestamp in item["date"].split(", "):
            date_label = timestamp[:10]
            if start_label <= date_label <= end_label:
                assert re.fullmatch(r"[0-9]{4}-[0-9]{2}-[0-9]{2} [0-9:]{8}", timestamp)
                for group_id in groups:
                    daily_counts[(group_id, date_label)] += 1
                    recorded_events += 1
index = pd.MultiIndex.from_product(
    [areas.index, pd.date_range(START, END, freq="D").strftime("%Y-%m-%d")],
    names=["group_id", "date"],
)
count_series = pd.Series(daily_counts)
panel = count_series.reindex(index, fill_value=0).rename("checkins").reset_index()
panel["date"] = pd.to_datetime(panel["date"])
panel = panel.merge(areas.reset_index(), on="group_id", validate="many_to_one")
panel["rate"] = panel["checkins"] / panel["n_businesses"]
del daily_counts, count_series, index


def add_calendar(frame):
    frame = frame.copy()
    frame["calendar_month"] = frame["date"].dt.month.astype(str)
    frame["day_of_week"] = frame["date"].dt.dayofweek.astype(str)
    return frame


panel = add_calendar(panel)
assert not panel.duplicated(["group_id", "date"]).any()
assert panel["checkins"].sum() == recorded_events
print(f"{len(panel):,} daily labels; {panel.checkins.eq(0).mean():.1%} zero-activity days")
display(panel[["metro", "zip", "category", "date", "checkins", "rate"]].head())

### 7. Daily weather và common coverage

Numeric finite với flag blank hoặc `T` (trace rain = 0); `A`/`S`/`M` là missing.
Đổi °F → °C, inch → mm. Chỉ train/chấm ngày đủ tmax, tmin, precipitation và có
`0 ≤ precipitation`, `tmin ≤ tmax`. Cả ba model dùng cùng các keys này; không
impute hoặc mượn normal để biến ngày thiếu weather thành observed day.


In [ ]:
observations = pd.read_csv(
    paths["observations"],
    usecols=["station_id", "date", "variable", "value", "unit", "flag"],
    dtype={"station_id": str, "flag": str},
    keep_default_na=False,
)
assert len(observations) == station_manifest["rows"]
assert not observations.duplicated(["station_id", "date", "variable"]).any()
observations = observations[observations["station_id"].isin(areas["station_id"])].copy()
observations["date"] = pd.to_datetime(observations["date"])
observations = observations[observations["date"] >= START]
assert observations["variable"].isin(["tmax", "tmin", "pcpn"]).all()
assert observations.loc[observations.variable.eq("pcpn"), "unit"].eq("inch").all()
assert observations.loc[~observations.variable.eq("pcpn"), "unit"].eq("degreeF").all()
observations["usable"] = pd.to_numeric(observations["value"], errors="coerce")
accepted = observations["flag"].str.strip().isin(["", "T"])
observations.loc[~accepted | ~np.isfinite(observations["usable"]), "usable"] = np.nan
weather_daily = observations.pivot(
    index=["station_id", "date"], columns="variable", values="usable"
).reset_index()
weather_daily["tmax_c"] = (weather_daily["tmax"] - 32) * 5 / 9
weather_daily["tmin_c"] = (weather_daily["tmin"] - 32) * 5 / 9
weather_daily["precip_mm"] = weather_daily["pcpn"] * 25.4
weather_daily["complete_weather"] = (
    np.isfinite(weather_daily[WEATHER]).all(axis=1)
    & weather_daily["precip_mm"].ge(0)
    & weather_daily["tmin_c"].le(weather_daily["tmax_c"])
)
weather_daily = weather_daily[["station_id", "date", *WEATHER, "complete_weather"]]
panel = panel.merge(weather_daily, on=["station_id", "date"], how="left", validate="many_to_one")
panel["complete_weather"] = panel["complete_weather"].fillna(False).astype(bool)
weather_coverage = (
    panel.assign(year=panel["date"].dt.year)
    .groupby(["year", "metro", "category"])
    .agg(total_days=("date", "size"), usable_days=("complete_weather", "sum"))
    .reset_index()
)
weather_coverage["usable_share"] = weather_coverage["usable_days"] / weather_coverage["total_days"]
model_panel = panel[panel["complete_weather"]].copy()
assert np.isfinite(model_panel[WEATHER]).all().all()
display(weather_coverage[weather_coverage["year"].ge(2018)].round(3))
print(f"Common model rows: {len(model_panel):,}/{len(panel):,}")

### 8. Ba model, chỉ nhận feature được cho phép

| Model | Cách học |
| --- | --- |
| calendar_mean | Mean training rate theo location/ngành/tháng, không đọc activity ngày trước |
| poisson_weather | Linear Poisson regression, weather + location + ngành + calendar |
| hgb_weather | Poisson loss, phi tuyến, 80 cây nhỏ, cùng input với Poisson |

Tọa độ + metro thay area one-hot trong learned models để giữ daily panel nhẹ.
Baseline lookup dùng area ID như metadata địa điểm. Tất cả preprocessing fit
training-only. `X` và `y` là object riêng, không truyền activity vào pipeline.
[Tài liệu sklearn: Poisson](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.PoissonRegressor.html),
[HGB](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.HistGradientBoostingRegressor.html).


In [ ]:
CAT_FEATURES = ["metro", "category", "calendar_month", "day_of_week"]
NUM_FEATURES = ["latitude", "longitude", *WEATHER]
FEATURES = ["area_id", *CAT_FEATURES, *NUM_FEATURES]
BASELINE_KEYS = ["area_id", "category", "calendar_month"]
MODEL_NAMES = ["calendar_mean", "poisson_weather", "hgb_weather"]
FORBIDDEN_INPUTS = {
    "checkins",
    "rate",
    "n_businesses",
    "train_checkins",
    "lag1",
    "lag12",
    "recent3",
    "trend_years",
}
assert not FORBIDDEN_INPUTS.intersection(FEATURES)


def feature_frame(frame):
    result = frame.loc[:, FEATURES].copy()
    assert set(result.columns) == set(FEATURES)
    assert np.isfinite(result[NUM_FEATURES]).all().all()
    return result


def make_model(name):
    preprocessing = ColumnTransformer(
        [
            ("numeric", StandardScaler(), NUM_FEATURES),
            ("category", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CAT_FEATURES),
        ],
        remainder="drop",
    )
    estimator = (
        PoissonRegressor(alpha=0.1, max_iter=500)
        if name == "poisson_weather"
        else HistGradientBoostingRegressor(
            loss="poisson",
            max_iter=80,
            max_leaf_nodes=15,
            min_samples_leaf=30,
            l2_regularization=1,
            early_stopping=False,
            random_state=SEED,
        )
    )
    return make_pipeline(preprocessing, estimator)


def fit_models(training):
    x_train = feature_frame(training)
    y_train = training["rate"].to_numpy()
    weights = training["n_businesses"].to_numpy()
    assert (y_train >= 0).all() and (y_train > 0).any()
    baseline = x_train[BASELINE_KEYS].copy()
    baseline["training_label"] = y_train
    fitted = {"calendar_mean": baseline.groupby(BASELINE_KEYS)["training_label"].mean()}
    for name in MODEL_NAMES[1:]:
        model = make_model(name)
        step = model.steps[-1][0]
        model.fit(x_train, y_train, **{f"{step}__sample_weight": weights})
        fitted[name] = model
    return fitted


def predict_rate(name, fitted, inputs):
    # Even if a caller adds activity columns, only the explicit input contract is read.
    x = feature_frame(inputs)
    if name == "calendar_mean":
        keys = pd.MultiIndex.from_frame(x[BASELINE_KEYS])
        prediction = fitted[name].reindex(keys).to_numpy()
    else:
        prediction = fitted[name].predict(x)
    if not np.isfinite(prediction).all() or (prediction < 0).any():
        raise ValueError("Unknown location/calendar key or invalid model output.")
    return np.maximum(prediction, 1e-8)


display(pd.Series(FEATURES, name="Allowed prediction input"))

## Results

### 9. Validation 2018: predict trước, activity truth dùng sau

Fit 2013–2017; predict bằng `X_valid` chỉ gồm allowed features. Sau khi có
prediction mới join actual activity để chấm. Primary: exposure-weighted rate
Poisson deviance từng metro/ngành, macro-average sáu segment ngang trọng số.
MAE rate giúp đọc sai số. Top-3 overlap là metric phụ, daily zeros/ties gây nhiễu;
đọc cùng random overlap, đặc biệt Nashville Spa chỉ có ít candidates.

Bỏ các ngày actual rate đồng nhất (gồm tất cả bằng 0) khỏi ranking metric vì
không có thứ hạng quan sát được; vẫn giữ chúng khi tính deviance/MAE. Segment
table report `ranking_days`/`total_days`. Ties còn lại dùng group_id cho cả
actual và prediction; top-k metric vẫn cần đọc thận trọng.


In [ ]:
def evaluate(fitted, evaluation, period):
    x_evaluation = feature_frame(evaluation)
    # Predict all models before reading evaluation activity labels.
    predictions = {name: predict_rate(name, fitted, x_evaluation) for name in MODEL_NAMES}
    prediction_tables, metric_rows = [], []
    for name, prediction in predictions.items():
        result = evaluation[
            [
                "group_id",
                "metro",
                "category",
                "city",
                "state",
                "zip",
                "date",
                "n_businesses",
                "checkins",
                "rate",
            ]
        ].copy()
        result["predicted_rate"] = prediction
        result["model"] = name
        result["period"] = period
        prediction_tables.append(result)
        for (metro, category), segment in result.groupby(["metro", "category"]):
            overlaps, chance_overlaps = [], []
            for _, day in segment.groupby("date"):
                if day["rate"].nunique() < 2:
                    continue  # Constant truth cannot identify a meaningful ranking.
                k = min(TOP_K, len(day))
                actual = set(
                    day.sort_values(["rate", "group_id"], ascending=[False, True]).head(k)[
                        "group_id"
                    ]
                )
                predicted = set(
                    day.sort_values(["predicted_rate", "group_id"], ascending=[False, True]).head(
                        k
                    )["group_id"]
                )
                overlaps.append(len(actual & predicted) / k)
                chance_overlaps.append(k / len(day))
            metric_rows.append(
                {
                    "period": period,
                    "model": name,
                    "metro": metro,
                    "category": category,
                    "poisson_deviance_rate": mean_poisson_deviance(
                        segment["rate"],
                        segment["predicted_rate"],
                        sample_weight=segment["n_businesses"],
                    ),
                    "mae_rate": mean_absolute_error(segment["rate"], segment["predicted_rate"]),
                    "top_k_overlap": np.mean(overlaps) if overlaps else np.nan,
                    "top_k_random_overlap": np.mean(chance_overlaps) if chance_overlaps else np.nan,
                    "ranking_days": len(overlaps),
                    "total_days": segment["date"].nunique(),
                    "n_areas": segment["group_id"].nunique(),
                    "rows": len(segment),
                    "zero_activity_share": segment["rate"].eq(0).mean(),
                }
            )
    segments = pd.DataFrame(metric_rows)
    macro = (
        segments.groupby(["period", "model"])[
            ["poisson_deviance_rate", "mae_rate", "top_k_overlap", "top_k_random_overlap"]
        ]
        .mean()
        .reset_index()
        .sort_values("poisson_deviance_rate")
    )
    return macro, segments, pd.concat(prediction_tables, ignore_index=True)


initial_training = model_panel[model_panel["date"] < VALIDATION_START]
validation = model_panel[
    model_panel["date"].between(VALIDATION_START, TEST_START - pd.Timedelta(days=1))
]
validation_models = fit_models(initial_training)
validation_metrics, validation_segments, validation_predictions = evaluate(
    validation_models, validation, "validation_2018"
)
BEST_MODEL = validation_metrics.iloc[0]["model"]
BEST_WEATHER_MODEL = validation_metrics[validation_metrics["model"].isin(MODEL_NAMES[1:])].iloc[0][
    "model"
]
display(validation_metrics.round(5))
print("Best overall:", BEST_MODEL, "| Default weather experiment:", BEST_WEATHER_MODEL)
print("Training:", initial_training.date.min().date(), "to", initial_training.date.max().date())

### 10. Refit đến hết 2018, test 2019

Giữ winner validation; không đổi winner khi đọc test. Không rolling update từ
activity 2019, không recursive feedback. Cùng input contract với ngày bất kỳ.


In [ ]:
final_training = model_panel[model_panel["date"] < TEST_START]
final_models = fit_models(final_training)
test = model_panel[model_panel["date"] >= TEST_START]
test_metrics, test_segments, test_predictions = evaluate(final_models, test, "test_2019")
display(test_metrics.round(5))
display(test_segments[test_segments["model"].eq(BEST_WEATHER_MODEL)].round(5))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
comparison = test_metrics.sort_values("poisson_deviance_rate")
axes[0].barh(comparison["model"], comparison["poisson_deviance_rate"], color="#28708d")
axes[0].invert_yaxis()
axes[0].set(
    xlabel="Macro rate Poisson deviance (lower better)", title="2019 daily conditional test"
)
axes[1].barh(comparison["model"], comparison["top_k_overlap"], color="#46977d")
axes[1].invert_yaxis()
axes[1].set(xlim=(0, 1), xlabel="Top-3 overlap (higher better)", title="Daily ZIP activity ranking")
fig.tight_layout()
plt.show()

### 11. Predict ngày bất kỳ với weather + địa điểm + ngành

`recommend(date, metro, category, weather=...)` nhận dict nhiệt độ °C, mưa mm
cho scenario chung của metro, hoặc DataFrame weather theo `station_id`. Cung
cấp weather thì ngày không bị giới hạn bởi snapshot hoặc lịch sử activity.
Bỏ `weather` để lấy observed ACIS đúng ngày; chỉ rank candidate có đủ weather
và report exclusions. Ngày thiếu toàn bộ weather yêu cầu nhập weather.

Default dùng weather model tốt nhất trên validation để explore phản ứng với
weather; bảng Results giữ baseline và winner toàn bộ riêng. `model="calendar_mean"`
cho phép đối chiếu baseline. Hàm chỉ đọc location metadata + weather; không đọc
panel activity, nhãn, số business hay trạng thái cửa hàng hiện tại.


In [ ]:
LOCATION_COLUMNS = [
    "group_id",
    "area_id",
    "metro",
    "city",
    "state",
    "zip",
    "category",
    "latitude",
    "longitude",
    "station_id",
]
locations = areas.reset_index()[LOCATION_COLUMNS].copy()
training_weather_range = final_training[WEATHER].agg(["min", "max"])


def predict_locations(date, metro, category, weather=None, model=None):
    target_date = pd.Timestamp(date)
    if pd.isna(target_date) or target_date.tzinfo is not None:
        raise ValueError("Use a valid timezone-naive calendar date, such as 2026-12-15.")
    target_date = target_date.normalize()
    name = BEST_WEATHER_MODEL if model is None else model
    if name not in MODEL_NAMES:
        raise ValueError(f"Choose model from {MODEL_NAMES}.")
    candidates = locations[locations["metro"].eq(metro) & locations["category"].eq(category)].copy()
    if candidates.empty:
        raise ValueError("No trained candidate for this metro/category; inspect cohort coverage.")
    candidate_count = len(candidates)
    candidates["date"] = target_date
    if weather is None:
        observed = weather_daily[weather_daily["date"].eq(target_date)]
        candidates = candidates.merge(
            observed[["station_id", *WEATHER, "complete_weather"]],
            on="station_id",
            how="left",
            validate="many_to_one",
        )
        candidates = candidates[candidates["complete_weather"].eq(True)].copy()
        source = "observed_acis"
    elif isinstance(weather, dict):
        if set(weather) != set(WEATHER):
            raise ValueError(f"Weather dict must contain exactly {WEATHER}.")
        for column in WEATHER:
            candidates[column] = float(weather[column])
        source = "supplied_weather"
    elif isinstance(weather, pd.DataFrame):
        if not {"station_id", *WEATHER}.issubset(weather.columns):
            raise ValueError("Weather table needs station_id plus the three weather variables.")
        if weather["station_id"].duplicated().any():
            raise ValueError("Weather table must have one row per station_id.")
        candidates = candidates.merge(
            weather[["station_id", *WEATHER]], on="station_id", how="left", validate="many_to_one"
        )
        if candidates[WEATHER].isna().any().any():
            raise ValueError("Supplied weather must cover all candidate stations.")
        source = "supplied_weather"
    else:
        raise ValueError("Weather must be a dict, a station DataFrame, or None for observations.")
    if candidates.empty:
        raise ValueError("No complete observed weather for that date; supply weather explicitly.")
    values = candidates[WEATHER].astype(float)
    if (
        not np.isfinite(values).all().all()
        or values["precip_mm"].lt(0).any()
        or values["tmin_c"].gt(values["tmax_c"]).any()
    ):
        raise ValueError("Weather must be finite, rain >= 0, and tmin <= tmax.")
    candidates[WEATHER] = values
    candidates = add_calendar(candidates)
    candidates["predicted_rate"] = predict_rate(name, final_models, feature_frame(candidates))
    candidates["model"] = name
    candidates["weather_source"] = source
    candidates["candidate_count"] = candidate_count
    candidates["weather_complete_candidates"] = len(candidates)
    candidates["weather_excluded_candidates"] = candidate_count - len(candidates)
    candidates["outside_training_weather_range"] = (
        (values < training_weather_range.loc["min"]) | (values > training_weather_range.loc["max"])
    ).any(axis=1)
    candidates = candidates.sort_values(["predicted_rate", "group_id"], ascending=[False, True])
    candidates["rank"] = np.arange(1, len(candidates) + 1)
    return candidates.reset_index(drop=True)


def recommend(date, metro="Tampa", category="Day Spas", weather=None, top_n=5, model=None):
    if not isinstance(top_n, int) or isinstance(top_n, bool) or top_n < 1:
        raise ValueError("top_n must be a positive integer.")
    ranking = predict_locations(date, metro, category, weather=weather, model=model)
    print(
        f"{ranking['date'].iloc[0].date()} | {ranking['model'].iloc[0]} | "
        f"weather={ranking['weather_source'].iloc[0]} | "
        f"{len(ranking)}/{ranking['candidate_count'].iloc[0]} candidates"
    )
    shortlist = ranking.head(top_n)
    display(shortlist[["rank", "city", "state", "zip", *WEATHER, "predicted_rate"]].round(4))
    return shortlist

### 12. Ba sample weather năm 2026 và một ngày tương lai

Observed samples dùng đúng **ngày** trong snapshot. Sample December nhập weather
để chứng minh API không cần activity của ngày trước, năm trước hay năm 2026.
Weather nhập là scenario, không được trình bày thành weather forecast thực tế.
Đổi ngày/metro/ngành/weather bên dưới để tự explore.


In [ ]:
SAMPLES = [
    {
        "sample": "philadelphia_2026_observed",
        "date": "2026-07-15",
        "metro": "Philadelphia",
        "category": "Restaurants",
    },
    {
        "sample": "nashville_2026_observed",
        "date": "2026-01-15",
        "metro": "Nashville",
        "category": "Restaurants",
    },
    {
        "sample": "tampa_spa_2026_observed",
        "date": "2026-09-15",
        "metro": "Tampa",
        "category": "Day Spas",
    },
    {
        "sample": "tampa_spa_future_scenario",
        "date": "2026-12-15",
        "metro": "Tampa",
        "category": "Day Spas",
        "weather": {"tmax_c": 27, "tmin_c": 19, "precip_mm": 2},
    },
]
sample_outputs = []
for specification in SAMPLES:
    arguments = {key: value for key, value in specification.items() if key != "sample"}
    ranking = predict_locations(**arguments)
    ranking["sample"] = specification["sample"]
    sample_outputs.append(ranking)
    recommend(**arguments, top_n=3)
sample_predictions = pd.concat(sample_outputs, ignore_index=True)
shortlists = sample_predictions[sample_predictions["rank"].le(TOP_K)].copy()
sample_summary = (
    sample_predictions.groupby("sample", sort=False)
    .agg(
        date=("date", "first"),
        metro=("metro", "first"),
        category=("category", "first"),
        model=("model", "first"),
        weather_source=("weather_source", "first"),
        candidates=("candidate_count", "first"),
        complete=("weather_complete_candidates", "first"),
        excluded=("weather_excluded_candidates", "first"),
        outside_training_share=("outside_training_weather_range", "mean"),
    )
    .reset_index()
)
display(sample_summary)

### 13. Kiểm tra input contract và lưu kết quả

Bỏ hoặc sửa activity truth không được đổi prediction. Kiểm tra ngày 2027 và
leap day 2028 với weather nhập, weather invalid bị từ chối, baseline không
phản ứng với weather. Lưu source/code/lockfile/output hashes để kiểm tra lại.


In [ ]:
assert len(test_predictions) == len(test) * len(MODEL_NAMES)
assert not test_predictions.duplicated(["model", "group_id", "date"]).any()
assert test_predictions["predicted_rate"].gt(0).all()
actual_keys = set(zip(test["group_id"], test["date"], strict=True))
for _, predicted in test_predictions.groupby("model"):
    assert set(zip(predicted["group_id"], predicted["date"], strict=True)) == actual_keys

probe = test.head(500).copy()
poisoned = probe.copy()
for column in FORBIDDEN_INPUTS:
    poisoned[column] = 999999
for name in MODEL_NAMES:
    expected = predict_rate(name, final_models, feature_frame(probe))
    assert np.allclose(predict_rate(name, final_models, poisoned), expected)
    no_activity = probe.drop(columns=list(FORBIDDEN_INPUTS), errors="ignore")
    assert np.allclose(predict_rate(name, final_models, no_activity), expected)
# Public inference payload and location catalog have no activity-derived columns.
assert not FORBIDDEN_INPUTS.intersection(locations.columns)
manual_weather = {"tmax_c": 27, "tmin_c": 19, "precip_mm": 2}
for date in ["2027-05-01", "2028-02-29"]:
    future = predict_locations(date, "Tampa", "Day Spas", weather=manual_weather)
    assert future["date"].eq(pd.Timestamp(date)).all() and len(future) == len(
        locations[locations.metro.eq("Tampa") & locations.category.eq("Day Spas")]
    )
    assert not FORBIDDEN_INPUTS.intersection(future.columns)
# Station-table weather input has the same result as a uniform weather dict.
station_weather = (
    locations[locations.metro.eq("Tampa") & locations.category.eq("Day Spas")][["station_id"]]
    .drop_duplicates()
    .assign(**manual_weather)
)
uniform = predict_locations("2026-12-15", "Tampa", "Day Spas", weather=manual_weather)
by_station = predict_locations("2026-12-15", "Tampa", "Day Spas", weather=station_weather)
assert np.allclose(uniform["predicted_rate"], by_station["predicted_rate"])
cold = predict_locations(
    "2026-12-15", "Tampa", "Day Spas", weather=manual_weather, model="calendar_mean"
)
hot = predict_locations(
    "2026-12-15",
    "Tampa",
    "Day Spas",
    weather={"tmax_c": 37, "tmin_c": 29, "precip_mm": 50},
    model="calendar_mean",
)
assert np.allclose(cold["predicted_rate"], hot["predicted_rate"])
for invalid in [
    {"tmax_c": 27, "tmin_c": 19, "precip_mm": -1},
    {"tmax_c": 10, "tmin_c": 19, "precip_mm": 2},
    {"tmax_c": np.nan, "tmin_c": 19, "precip_mm": 2},
]:
    try:
        predict_locations("2026-12-15", "Tampa", "Day Spas", weather=invalid)
    except ValueError:
        pass
    else:
        raise AssertionError("Invalid weather accepted.")
try:
    predict_locations("2030-01-01", "Tampa", "Day Spas")
except ValueError:
    pass
else:
    raise AssertionError("Missing future weather should require supplied input.")

chosen_predictions = test_predictions[test_predictions["model"].eq(BEST_WEATHER_MODEL)]
recomputed = [
    mean_poisson_deviance(
        segment["rate"], segment["predicted_rate"], sample_weight=segment["n_businesses"]
    )
    for _, segment in chosen_predictions.groupby(["metro", "category"])
]
assert np.isclose(
    np.mean(recomputed),
    test_metrics.set_index("model").loc[BEST_WEATHER_MODEL, "poisson_deviance_rate"],
)
RUN_DIR.mkdir(parents=True, exist_ok=False)
exports = {
    "cohort_areas.csv": areas.reset_index(),
    "daily_panel.csv": panel[
        [
            "group_id",
            "date",
            "checkins",
            "rate",
            "n_businesses",
            *FEATURES,
            "station_id",
            "complete_weather",
        ]
    ],
    "weather_coverage.csv": weather_coverage,
    "validation_metrics.csv": validation_metrics,
    "validation_segments.csv": validation_segments,
    "test_metrics.csv": test_metrics,
    "test_segments.csv": test_segments,
    "validation_predictions.csv": validation_predictions,
    "test_predictions.csv": test_predictions,
    "sample_predictions.csv": sample_predictions,
    "sample_summary.csv": sample_summary,
    "shortlists.csv": shortlists,
}
for filename, frame in exports.items():
    frame.to_csv(RUN_DIR / filename, index=False)
manifest = {
    "status": "complete",
    "created_at_utc": datetime.now(UTC).isoformat(),
    "purpose": "Daily conditional prediction POC; weather/location/industry/date inputs",
    "source_hashes": source_hashes,
    "source_paths": {key: str(value.relative_to(ROOT)) for key, value in paths.items()},
    "category_parser_sha256": sha256(ROOT / "src/sitesense/import_data/yelp.py"),
    "notebook_sha256": sha256(SOURCE_NOTEBOOK),
    "lockfile_sha256": sha256(ROOT / "uv.lock"),
    "versions": versions,
    "features": FEATURES,
    "learned_features": CAT_FEATURES + NUM_FEATURES,
    "baseline_keys": BASELINE_KEYS,
    "selected_model": BEST_MODEL,
    "selected_weather_model": BEST_WEATHER_MODEL,
    "configuration": {
        "metro_counties": METRO_COUNTIES,
        "categories": CATEGORIES,
        "start": str(START.date()),
        "cohort_end": str(COHORT_END.date()),
        "validation_start": str(VALIDATION_START.date()),
        "test_start": str(TEST_START.date()),
        "end": str(END.date()),
        "min_businesses": MIN_BUSINESSES,
        "min_train_checkins": MIN_TRAIN_CHECKINS,
        "top_k": TOP_K,
        "seed": SEED,
    },
    "training_end": str(final_training["date"].max().date()),
    "model_parameters": {
        name: final_models[name].steps[-1][1].get_params() for name in MODEL_NAMES[1:]
    },
    "cohort_groups": len(areas),
    "business_category_rows": len(cohort),
    "panel_rows": len(panel),
    "model_rows": len(model_panel),
    "samples": SAMPLES,
    "weather_policy": "Observed days for evaluation; supplied weather for arbitrary dates",
    "activity_policy": "Cohort/labels/weights/scoring only; no activity features",
    "timestamp_policy": "Source date labels; timezone and station window unverified",
    "activity_2026_validation": "Unavailable",
    "output_sha256": {filename: sha256(RUN_DIR / filename) for filename in exports},
}
(RUN_DIR / "manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2) + "\n")
print("All input-contract and reconciliation checks passed. Artifacts:", RUN_DIR.relative_to(ROOT))

## Takeaways

- Một experiment daily duy nhất, cùng input contract từ validation đến inference:
  **weather + địa điểm + ngành + ngày**, không activity features. Actual activity
  được dùng làm nhãn để học/chấm; bỏ truth vẫn predict được.
- Đọc winner toàn bộ và weather model riêng. Baseline tốt hơn thì không tuyên bố
  weather tăng chất lượng; sample chỉ chứng minh đường input → model → ranking.
- Mọi ngày đều nhận được nếu cung cấp weather hợp lệ. Với weather observed, chỉ
  rank những candidate đủ weather đúng ngày và report các candidate bị loại.
- Output mô tả recorded activity của cohort lịch sử. Chưa có actual 2026 để đo
  accuracy 2026, chưa có nhãn thành công cửa hàng mới hoặc spatial holdout.
- Daily zeros/ties, source-date alignment, station missingness và snapshot bias
  làm ranking khó đọc. Dùng shortlist để explore, cần kiểm tra thực địa trước
  quyết định mở cửa hàng. Notebook không thay mock forecasting screen trong app.

Quan sát mặc định: baseline deviance test **0.05664**, HGB **0.05880**, Poisson
**0.08884**. HGB weather có test MAE **0.04297 check-in/business/ngày**; chưa tốt
hơn baseline. Tất cả 13 code cells đã chạy; mutation/drop-label, future-date,
leap-day, weather validity và station-input checks đều pass.
